# Week 6 — K8s Health MCP

Mock Kubernetes health exposed through **MCP**, used from the **OpenAI Agents SDK**. **No real cluster** is required.

**Core files:** `sample_cluster.json`, `server.py`, `agent_client.py`, `requirements.txt`. **Optional:** `agent_run_example.png`, `kind_kubectl_learning_ns.png`.

**Tip:** set the kernel working directory to **this folder** (same directory as `agent_client.py`) so paths and imports behave as documented.

## Mock data vs a live cluster

| Source | What the MCP uses |
|--------|-------------------|
| `sample_cluster.json` | **Only** this file. Namespaces **`payments`** and **`platform`**. |
| kind / cloud cluster | **Not** read by `server.py` unless you add a live mode. |

If you create **`learning-ns`** on a real cluster and ask the agent about it, you still get “not in mock data” until you extend the JSON or wire `kubectl` (or the K8s client). That is **expected**.

## What this demonstrates

1. **Tools:** `cluster_overview`, `namespace_health`, `workload_health`, `incident_summary`  
2. **Resource:** `runbook://k8s/common-issues`  
3. **Stdio MCP:** `agent_client.py` starts `server.py` via `MCPServerStdio` — do **not** run `server.py` yourself for the normal flow.

## Files in this folder

| File | Role |
|------|------|
| `sample_cluster.json` | Mock nodes, namespaces, pods, deployments |
| `server.py` | FastMCP server (tools + runbook) |
| `agent_client.py` | Agent + MCP client; loads **repository root** `.env` for `OPENAI_API_KEY` (same rule as below) |
| `requirements.txt` | `mcp`, `openai-agents`, `python-dotenv` |
| `README.md` | Short pointer + file index |
| `week6_k8s_health_mcp.ipynb` | This walkthrough |
| `agent_run_example.png` | Example terminal: MCP run vs `learning-ns` |
| `kind_kubectl_learning_ns.png` | Example kind/`kubectl`: nginx OK, `broken-pod` ImagePullBackOff |

## Screenshots

Run the next cell to render the PNGs (requires `IPython`; standard in Jupyter).

In [ ]:
from pathlib import Path

from IPython.display import Image, Markdown, display

here = Path.cwd()
for filename, caption in [
    ("agent_run_example.png", "MCP agent run (mock has no `learning-ns`)."),
    ("kind_kubectl_learning_ns.png", "kind + `kubectl`: `learning-ns`, working nginx, `broken-pod` ImagePullBackOff."),
]:
    display(Markdown(f"### {caption}\n`{filename}`"))
    p = here / filename
    if p.is_file():
        display(Image(filename=str(p)))
    else:
        print(f"Not found: {p.resolve()}")

## 1 — Prerequisites

- Python **3.10+**
- **`OPENAI_API_KEY`** in the environment or in **`.env`** at the **repository root** (folder that contains `6_mcp/`; three levels up from `SamuelAdebodun/`)
- **`pip`** or **`uv`**

**Optional (practice only):** Docker + WSL integration, `kubectl`, kind — unrelated to the mock MCP.

On Windows, running Week 6 from **WSL** is usually the least painful.

## 2 — Install (terminal in this directory)

```bash
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install -r requirements.txt
# or: uv pip install -r requirements.txt
```

If the key is not in that `.env`, export `OPENAI_API_KEY` in the shell before the next step.

## 3 — Run the agent (terminal)

Do **not** start `server.py` first; the client spawns it.

```bash
python agent_client.py "Call cluster_overview, then namespace_health for payments."
```

With **uv**, run from **this** folder only (otherwise `uv run` may sync the whole repo):

```bash
uv run python agent_client.py "Call cluster_overview, then namespace_health for payments."
```

**Prompts that match the mock:** anything under **`payments`** / **`platform`** (e.g. workload `payments-api`, `incident_summary` for `platform`).

## 4 — Optional: try the agent from this notebook

Loads the same **repository root** `.env` as `agent_client.py` (`parents[2]` from this folder), then runs `agent_client.py` once. Skip if you prefer the terminal only.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv

here = Path.cwd()
if not (here / "agent_client.py").is_file():
    raise RuntimeError(
        "Set the notebook kernel cwd to the SamuelAdebodun folder (where agent_client.py lives)."
    )

_repo_root = here.parents[2]
load_dotenv(_repo_root / ".env", override=False)
load_dotenv(override=False)

if not os.environ.get("OPENAI_API_KEY"):
    print("No OPENAI_API_KEY — add .env at repo root (parent of 6_mcp) or export in your environment.")
else:
    cmd = [
        sys.executable,
        str(here / "agent_client.py"),
        "Call cluster_overview briefly.",
    ]
    print("Running:", " ".join(cmd[:2]), "...")
    subprocess.run(cmd, cwd=here, check=False)

## 5 — Optional: kind + `kubectl` (practice)

Independent of the MCP:

1. Docker Desktop → **WSL integration** on; `docker ps` works without `sudo`.
2. Install `kubectl` and `kind` in WSL → `kind create cluster --name local-wsl-k8s` → `kubectl get nodes`.
3. `kubectl apply -f wsl-workload.yaml` → `kubectl get all -n learning-ns`.

Smoke: `kubectl port-forward -n learning-ns deploy/working-nginx 8080:80` → http://localhost:8080

## Troubleshooting

| Symptom | What to check |
|---------|----------------|
| `ModuleNotFoundError: mcp` / `agents` | `pip install -r requirements.txt` in the same venv you use to run the client |
| `OPENAI_API_KEY must be set` | Repository root `.env` or shell export |
| Agent: `learning-ns` not in data | Expected for mock-only; extend JSON or add live reads |
| Docker in WSL | Docker Desktop running; integration enabled |
| `uv run` installs too much | Use venv + `python agent_client.py` from this folder |

## Extending to a real cluster

Add a **mock / live** switch in `server.py` (e.g. env var). Live path: `kubectl … -o json` or the official Kubernetes Python client; keep **identical tool names**. Prefer read-only RBAC; ensure `kubeconfig`’s `server:` URL is reachable from the machine running the MCP.